In [0]:
%sql
-- Create (or replace) a Payer 360 table from Komodo plans
CREATE or replace temp view payer_base AS
SELECT DISTINCT
    PAYER_ID,
    PAYER_NAME,
    PARENT_ID,
    PARENT_NAME,
    KH_PLAN_ID          AS PLAN_ID,
    PBM_PROCESSOR,
    INSURANCE_GROUP,
    INSURANCE_SEGMENT
FROM com_edp_prd.com_raw.kom_plans
WHERE PAYER_ID IS NOT NULL;
select * from payer_base


In [0]:
%sql
-- =====================================================================
-- Payer 360 | Supporting temp views (grouped / de-cluttered)
-- =====================================================================

-- ---------------------------------------------------------------------
-- 1) Elaprase treatment events (Medical + Pharmacy) used for patient/HCP metrics
-- ---------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE AS
SELECT *
FROM (
    -- Medical (Elaprase NDC)
    SELECT DISTINCT
        PATIENT_ID                                  AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI)      AS HCP_NPI,
        NDC11                                       AS CODE,
        MEDICAL_EVENT_ID                            AS EVENT_ID,
        SERVICE_DATE                                AS FILL_DATE,
        PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
        KH_PLAN_ID                                  AS KH_PLAN,
        null                                        AS PHARMACY_CHANNEL,
        'MEDICAL_EVENTS'                            AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    -- Pharmacy (Elaprase NDC | Paid only)
    SELECT DISTINCT
        PATIENT_ID                                  AS PATIENT_ID,
        PRESCRIBER_NPI                              AS HCP_NPI,
        NDC11                                       AS CODE,
        PHARMACY_EVENT_ID                           AS EVENT_ID,
        FILL_DATE                                   AS FILL_DATE,
        NULL                                        AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        PHARMACY_CHANNEL                            AS PHARMACY_CHANNEL,
        'PHARMACY_EVENTS'                           AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    -- Medical (Elaprase Procedure codes)
    SELECT DISTINCT
        PATIENT_ID                                  AS PATIENT_ID,
        RENDERING_NPI                               AS HCP_NPI,
        PROCEDURE_CODE                              AS CODE,
        MEDICAL_EVENT_ID                            AS EVENT_ID,
        SERVICE_DATE                                AS FILL_DATE,
        PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
        KH_PLAN_ID                                  AS KH_PLAN,
        null                                        AS PHARMACY_CHANNEL,
        'MEDICAL_EVENTS'                            AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    )
) t
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';


-- ---------------------------------------------------------------------
-- 2) Total lives base (ALL medical events + PAID pharmacy events)
-- ---------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW total_lives AS
SELECT DISTINCT
    PATIENT_ID                                  AS total_lives,
    COALESCE(RENDERING_NPI, REFERRING_NPI)      AS HCP_NPI,
    MEDICAL_EVENT_ID                            AS EVENT_ID,
    SERVICE_DATE                                AS FILL_DATE,
    PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
    KH_PLAN_ID                                  AS KH_PLAN,
    null                                        AS PHARMACY_CHANNEL,
    'MEDICAL_EVENTS'                            AS TABLE_NAME
FROM com_edp_prd.com_raw.kom_medical_events
WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-11-30'

UNION

SELECT DISTINCT
    PATIENT_ID                                  AS total_lives,
    PRESCRIBER_NPI                              AS HCP_NPI,
    PHARMACY_EVENT_ID                           AS EVENT_ID,
    FILL_DATE                                   AS FILL_DATE,
    NULL                                        AS PLACE_OF_SERVICE,
    COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
    PHARMACY_CHANNEL                            AS PHARMACY_CHANNEL, 
    'PHARMACY_EVENTS'                           AS TABLE_NAME
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';


-- ---------------------------------------------------------------------
-- 3) Plan-level aggregations
-- ---------------------------------------------------------------------
-- 3a) Total Elaprase patients (5yr window in your base)
CREATE OR REPLACE TEMP VIEW plan_patient_counts AS
SELECT DISTINCT
    KH_PLAN AS PLAN_ID,
    COUNT(DISTINCT PATIENT_ID) AS total_patients
FROM MPSII_TREATMENT_TABLE
WHERE KH_PLAN IS NOT NULL
GROUP BY KH_PLAN;

-- 3b) Total lives (distinct patients with ANY medical OR paid pharmacy event)
CREATE OR REPLACE TEMP VIEW plan_total_lives AS
SELECT DISTINCT
    KH_PLAN AS PLAN_ID,
    COUNT(DISTINCT total_lives) AS total_lives
FROM total_lives
WHERE KH_PLAN IS NOT NULL
GROUP BY KH_PLAN;

-- 3c) New Elaprase patients (Quarter, month window)
CREATE OR REPLACE TEMP VIEW plan_new_elaprase_patients AS
SELECT DISTINCT
    PLAN_ID,

    /* New in Quarter */
    COUNT(DISTINCT CASE
        WHEN FIRST_FILL_DATE BETWEEN QUARTER_START AND END_DATE
        THEN PATIENT_ID
    END) AS new_elaprase_patients_qtr,

    /* New in Month */
    COUNT(DISTINCT CASE
        WHEN FIRST_FILL_DATE BETWEEN MONTH_START AND END_DATE
        THEN PATIENT_ID
    END) AS new_elaprase_patients_mth

FROM (
    SELECT DISTINCT
        PATIENT_ID,
        KH_PLAN AS PLAN_ID,

        /* First-ever Elaprase fill per patient */
        MIN(FILL_DATE) OVER (PARTITION BY PATIENT_ID) AS FIRST_FILL_DATE,

        /* Dynamic windows derived from end date */
        DATE_TRUNC('QUARTER', DATE('2025-11-30')) AS QUARTER_START,
        DATE_TRUNC('MONTH',   DATE('2025-11-30')) AS MONTH_START,
        DATE('2025-11-30')                        AS END_DATE

    FROM MPSII_TREATMENT_TABLE
    WHERE KH_PLAN IS NOT NULL
) t
GROUP BY PLAN_ID;


-- 3d) Total HCPs (distinct treating/prescribing NPIs)
CREATE OR REPLACE TEMP VIEW plan_total_hcps AS
SELECT DISTINCT
    KH_PLAN AS PLAN_ID,
    COUNT(DISTINCT HCP_NPI) AS total_hcps
FROM MPSII_TREATMENT_TABLE
WHERE KH_PLAN IS NOT NULL
  AND HCP_NPI IS NOT NULL
GROUP BY KH_PLAN;

-- 3e) Total HCOs (to be added)

-- 3f) Payer Ranking (on the basis of total lives; Dense Ranking- same ranking to same total lives but the sequence will continue)
CREATE OR REPLACE TEMP VIEW payer_total_lives_rank AS
SELECT DISTINCT
    pb.PAYER_ID,
    pb.PAYER_NAME,

    /* True distinct lives at payer level */
    COUNT(DISTINCT tl.total_lives) AS payer_total_lives,

    /* Rank by payer size */
    DENSE_RANK() OVER (
        ORDER BY COUNT(DISTINCT tl.total_lives) DESC
    ) AS payer_rank

FROM total_lives tl
JOIN payer_base pb
  ON tl.KH_PLAN = pb.PLAN_ID
WHERE tl.KH_PLAN IS NOT NULL
GROUP BY
    pb.PAYER_ID,
    pb.PAYER_NAME;




In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payer_base_patient_lives_v2 AS
SELECT DISTINCT
    a.*,
    t.PHARMACY_CHANNEL AS PHARMACY_CHANNEL,
    COALESCE(l.total_lives, 0)           AS total_lives,
    COALESCE(p.total_patients, 0)        AS total_elaprase_patients,
    COALESCE(n.new_elaprase_patients_qtr, 0) AS new_elaprase_patients_qtr,
    COALESCE(n.new_elaprase_patients_mth, 0) AS new_elaprase_patients_mth,
    COALESCE(h.total_hcps, 0)            AS total_hcps,
--- COALESCE(hco.total_hcos, 0)          AS total_hcos
    CASE
      WHEN rank.payer_rank IS NULL THEN 0
      ELSE rank.payer_rank
    END AS payer_rank
FROM payer_base a
LEFT JOIN plan_patient_counts p
    ON a.PLAN_ID = p.PLAN_ID
LEFT JOIN plan_total_lives l
    ON a.PLAN_ID = l.PLAN_ID
LEFT JOIN plan_new_elaprase_patients n
    ON a.PLAN_ID = n.PLAN_ID
LEFT JOIN plan_total_hcps h
    ON a.PLAN_ID = h.PLAN_ID
-- LEFT JOIN payer_base_hcos hco
--  ON a.PLAN_ID = hco.PLAN_ID
LEFT JOIN payer_total_lives_rank rank
    ON a.PAYER_ID = rank.PAYER_ID
LEFT JOIN MPSII_TREATMENT_TABLE t
    ON a.PLAN_ID = t.KH_PLAN
;

SELECT DISTINCT * FROM payer_base_patient_lives_v2;

In [0]:
%sql
-- Elaprase MEDICAL claims (distinct medical event ids)
CREATE OR REPLACE TEMP VIEW elaprase_med_claims AS
SELECT DISTINCT
  KH_PLAN_ID AS PLAN_ID,
  MEDICAL_EVENT_ID AS EVENT_ID
FROM com_edp_prd.com_raw.kom_medical_events
WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-11-30'
  AND (
        NDC11 IN ('54092070001','540920700')
        OR PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                              '38206','38230','38232','38240','38241','38242','38243','38250')
      )
  AND KH_PLAN_ID IS NOT NULL;


-- Elaprase PHARMACY claims (distinct pharmacy event ids)
CREATE OR REPLACE TEMP VIEW elaprase_pharmacy_claims AS
SELECT DISTINCT
  COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS PLAN_ID,
  PHARMACY_EVENT_ID AS EVENT_ID,
  TRANSACTION_RESULT
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30'
  AND NDC11 IN ('54092070001','540920700')
  AND COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) IS NOT NULL;


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW plan_total_pharmacy_claims AS
SELECT DISTINCT
  PLAN_ID,
  COUNT(DISTINCT EVENT_ID) AS total_pharmacy_claims
FROM elaprase_pharmacy_claims
GROUP BY PLAN_ID;

CREATE OR REPLACE TEMP VIEW plan_approved_fills AS
SELECT DISTINCT
  PLAN_ID,
  COUNT(DISTINCT EVENT_ID) AS approved_fills
FROM elaprase_pharmacy_claims
WHERE UPPER(TRANSACTION_RESULT) = 'PAID'
GROUP BY PLAN_ID;


CREATE OR REPLACE TEMP VIEW plan_rejected_fills AS
SELECT DISTINCT
  PLAN_ID,
  COUNT(DISTINCT EVENT_ID) AS rejected_fills
FROM elaprase_pharmacy_claims
WHERE UPPER(TRANSACTION_RESULT) = 'REJECTED'
GROUP BY PLAN_ID;


CREATE OR REPLACE TEMP VIEW plan_reversed_fills AS
SELECT DISTINCT
  PLAN_ID,
  COUNT(DISTINCT EVENT_ID) AS reversed_fills
FROM elaprase_pharmacy_claims
WHERE UPPER(TRANSACTION_RESULT) = 'REVERSED'
GROUP BY PLAN_ID;

CREATE OR REPLACE TEMP VIEW plan_elaprase_rejection_rate AS
SELECT DISTINCT
  t.PLAN_ID,
  COALESCE(r.rejected_fills, 0)        AS rejected_fills,
  COALESCE(t.total_pharmacy_claims, 0) AS total_pharmacy_claims,
  CASE
    WHEN COALESCE(t.total_pharmacy_claims, 0) = 0 THEN 0
    ELSE COALESCE(r.rejected_fills, 0) * 1.0
         / t.total_pharmacy_claims
  END AS elaprase_rejection_rate
FROM plan_total_pharmacy_claims t
LEFT JOIN plan_rejected_fills r
  ON t.PLAN_ID = r.PLAN_ID;



In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payer_base_patient_lives_v3 AS
SELECT DISTINCT
  a.*,
  COALESCE(tc.total_pharmacy_claims, 0)           AS total_claims,
  COALESCE(ap.approved_fills, 0)         AS approved_fills,
  COALESCE(rj.rejected_fills, 0)         AS rejected_fills,
  COALESCE(rv.reversed_fills, 0)         AS reversed_fills,
  COALESCE(dr.elaprase_rejection_rate, 0)   AS elaprase_rejection_rate

FROM payer_base_patient_lives_v2 a
LEFT JOIN plan_total_pharmacy_claims tc
  ON a.plan_id = tc.plan_id
LEFT JOIN plan_approved_fills ap
  ON a.plan_id = ap.plan_id
LEFT JOIN plan_rejected_fills rj
  ON a.plan_id = rj.plan_id
LEFT JOIN plan_reversed_fills rv
  ON a.plan_id = rv.plan_id
LEFT JOIN plan_elaprase_rejection_rate dr
  ON a.plan_id = dr.plan_id
WHERE PHARMACY_CHANNEL IS NOT NULL  
AND UPPER(PHARMACY_CHANNEL) != 'UNKNOWN';


SELECT DISTINCT * FROM payer_base_patient_lives_v3;


In [0]:
%sql
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer360_master AS
SELECT DISTINCT 

-- PAYER IDENTITY
v3.PAYER_ID                AS PAYER_ID,
v3.PAYER_NAME              AS PAYER_NAME,
v3.PAYER_RANK              AS PAYER_RANK,
v3.PARENT_ID               AS PARENT_ID,
v3.PARENT_NAME             AS PARENT_NAME,
v3.PLAN_ID                 AS PLAN_ID,
v3.PBM_PROCESSOR           AS PBM_PROCESSOR,
v3.INSURANCE_GROUP         AS INSURANCE_GROUP,
v3.INSURANCE_SEGMENT       AS INSURANCE_SEGMENT,
v3.PHARMACY_CHANNEL        AS PHARMACY_CHANNEL,

-- PATIENT & PROVIDER COUNTS (CORE VISIBILITY)
v3.TOTAL_LIVES             AS TOTAL_LIVES,
v3.TOTAL_ELAPRASE_PATIENTS AS TOTAL_ELAPRASE_PATIENTS,
v3.NEW_ELAPRASE_PATIENTS_QTR AS NEW_ELAPRASE_PATIENTS_QTR,
v3.NEW_ELAPRASE_PATIENTS_MTH AS NEW_ELAPRASE_PATIENTS_MTH,
v3.TOTAL_HCPS              AS TOTAL_HCPS,
-- v3.TOTAL_HCOS              AS TOTAL_HCOS,

-- CLAIMS & FUNNEL METRICS
v3.TOTAL_CLAIMS            AS TOTAL_CLAIMS,
v3.APPROVED_FILLS          AS APPROVED_FILLS,
v3.REJECTED_FILLS          AS REJECTED_FILLS,
v3.REVERSED_FILLS          AS REVERSED_FILLS,
v3.ELAPRASE_REJECTION_RATE AS ELAPRASE_REJECTION_RATE,

-- FIELD FORCE & ENGAGEMENT--- Dummy placeholders (populate later)

  -- Dummy placeholders (populate later)
  CAST(NULL AS INT)    AS total_hcos        -- count
  -- CAST(NULL AS STRING) AS pharmacy_channel,  -- retail/specialty/ltc
  -- CAST(NULL AS INT)    AS payer_ranking      -- rank

FROM payer_base_patient_lives_v3 v3;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payer_name_mapping AS
SELECT
  PAYER_NAME,

  CASE
    /* United / Optum */
    WHEN UPPER(PAYER_NAME) LIKE '%UNITED%' 
      OR UPPER(PAYER_NAME) LIKE '%OPTUM%' 
      OR UPPER(PAYER_NAME) LIKE '%UHC%'
      THEN 'United / Optum / Emisar'

    /* Anthem / Elevance */
    WHEN UPPER(PAYER_NAME) LIKE '%ANTHEM%'
      OR UPPER(PAYER_NAME) LIKE '%WELLPOINT%'
      THEN 'Anthem / Elevance / Carelon'

    /* Aetna / CVS */
    WHEN UPPER(PAYER_NAME) LIKE '%AETNA%'
      OR UPPER(PAYER_NAME) LIKE '%CVS%'
      OR UPPER(PAYER_NAME) LIKE '%SILVERSCRIPT%'
      THEN 'Aetna / CVS / Zinc'

    /* Cigna / Express Scripts */
    WHEN UPPER(PAYER_NAME) LIKE '%EXPRESS SCRIPTS%'
      THEN 'Cigna / ESI / Evernorth'

    /* Prime / HCSC */
    WHEN UPPER(PAYER_NAME) LIKE '%PRIME THERAPEUTICS%'
      OR UPPER(PAYER_NAME) LIKE '%HCSC%'
      THEN 'Prime Therapeutics / HCSC'

    /* Molina */
    WHEN UPPER(PAYER_NAME) LIKE '%MOLINA%'
      THEN 'Molina'

    /* Centene */
    WHEN UPPER(PAYER_NAME) LIKE '%WELLCARE%'
      OR UPPER(PAYER_NAME) LIKE '%MERIDIAN%'
      THEN 'Centene'

    /* Humana */
    WHEN UPPER(PAYER_NAME) LIKE '%HUMANA%'
      THEN 'Humana'

    /* Kaiser */
    WHEN UPPER(PAYER_NAME) LIKE '%KAISER%'
      THEN 'Kaiser'

    /* Tricare */
    WHEN UPPER(PAYER_NAME) LIKE '%TRICARE%'
      THEN 'Tricare'

    /* Navitus */
    WHEN UPPER(PAYER_NAME) LIKE '%NAVITUS%'
      THEN 'Navitus'

    /* BCBS – State-based mapping */
    WHEN UPPER(PAYER_NAME) LIKE '%BLUECROSS%'
      OR UPPER(PAYER_NAME) LIKE '%BLUE CROSS%'
      OR UPPER(PAYER_NAME) LIKE '%BCBS%'
      THEN 'Prime Therapeutics / HCSC'

    /* Default */
    ELSE 'UNMAPPED'
  END AS PAYER_ACCOUNT_NAME

FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master;


In [0]:
%sql
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer360_master AS
SELECT DISTINCT
  p.*,
  i.PIE_COMPLETED,
  i.ACCOUNT_DIRECTOR
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master p
LEFT JOIN payer_name_mapping m
  ON p.PAYER_NAME = m.PAYER_NAME
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_info i
  ON m.PAYER_ACCOUNT_NAME = i.PAYER_ACCOUNT_NAME;


In [0]:
%sql
SELECT * 
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master;